In [9]:
from pathlib import Path
_template = str(Path(__vsc_ipynb_file__).parent.parent / '../template.ipynb')
%run "$_template"

In [10]:
import lightgbm as lgb

In [11]:
df_test = pd.read_csv("../../data/modeling/test.csv")

photo_litho = ['UV_type', 'Resolution', 'Energy_Exposure', 'Line_CD']
X = df_test[photo_litho].copy()
y = df_test['is_low_yield'].copy()

# 모델 학습 시와 동일하게 UV_type을 category dtype으로 변환
X['UV_type'] = X['UV_type'].astype('category')

In [17]:
# =========================
# 1. 모델 불러오기
# =========================
photo_litho_model = lgb.Booster(model_file="../../model/photo_litho_lgbm.txt")

# =========================
# 2. 불량 예측 확률 계산
# =========================
bad_prob = photo_litho_model.predict(X)

result = df_test[photo_litho].copy()
result["bad_prob"] = bad_prob
result["y_true"] = y.values if hasattr(y, "values") else y

# =========================
# 3. 위험구간 기준 설정
# =========================
low_threshold = 0.3
high_ratio = 0.1

high_threshold = result["bad_prob"].quantile(1 - high_ratio)

print("저위험 기준 bad_prob <=", low_threshold)
print("고위험 기준 bad_prob >=", high_threshold)

# =========================
# 4. 위험구간 부여
# =========================
def assign_risk_group(prob):
    if prob <= low_threshold:
        return "저위험"
    elif prob >= high_threshold:
        return "고위험"
    else:
        return "중위험"

result["risk_group"] = result["bad_prob"].apply(assign_risk_group)

# =========================
# 5. 위험구간별 성능 요약
# =========================
risk_summary = (
    result
    .groupby("risk_group")
    .agg(
        data_count=("y_true", "count"),
        actual_defect_count=("y_true", "sum"),
        actual_defect_rate=("y_true", "mean"),
        mean_pred_prob=("bad_prob", "mean"),
        min_pred_prob=("bad_prob", "min"),
        max_pred_prob=("bad_prob", "max")
    )
    .reset_index()
)

risk_summary["data_ratio_percent"] = risk_summary["data_count"] / len(result) * 100
risk_summary["actual_defect_rate_percent"] = risk_summary["actual_defect_rate"] * 100
risk_summary["mean_pred_prob_percent"] = risk_summary["mean_pred_prob"] * 100

display(risk_summary)

# =========================
# 6. 결과 확인
# =========================
display(result[["bad_prob", "y_true", "risk_group"]].head())

저위험 기준 bad_prob <= 0.3
고위험 기준 bad_prob >= 0.8739555099298743


,risk_group,data_count,actual_defect_count,actual_defect_rate,mean_pred_prob,min_pred_prob,max_pred_prob,data_ratio_percent,actual_defect_rate_percent,mean_pred_prob_percent
0,고위험,462,446,0.965368,0.928280,0.873966,0.982771,10.006498,96.536797,92.827959
1,저위험,3799,2,0.000526,0.063335,0.000157,0.299800,82.282868,0.052645,6.333467
2,중위험,356,136,0.382022,0.616812,0.300047,0.873948,7.710635,38.202247,61.681208


,bad_prob,y_true,risk_group
0,0.001136,0,저위험
1,0.024555,0,저위험
2,0.870281,1,중위험
3,0.127749,0,저위험
4,0.035529,0,저위험
